# 4062. Transform Array Using Pair Operations — Nhập môn kỹ thuật "Tìm bất biến" (Invariant)

Bài toán LeetCode (Medium):

```python
class Solution:
    def canTransform(self, source: list[int], target: list[int]) -> bool:
        ...
```

## Mục tiêu của notebook

Sau notebook này, bạn sẽ hiểu:

- Đọc và **mô phỏng** một phép biến đổi lạ bằng code Python.
- Cách dùng **thí nghiệm ngẫu nhiên** (`random`) để đoán ra quy luật.
- Khái niệm **bất biến (invariant)**: đại lượng *không bao giờ thay đổi* dù ta làm phép toán bao nhiêu lần.
- Vì sao "tổng không đổi" là **điều kiện cần**, và vì sao nó cũng là **điều kiện đủ** (kỹ thuật "chọn một phần tử làm neo").
- Cách kiểm chứng bằng **BFS vét cạn** trên miền nhỏ (lời giải "thô" để đối chiếu).
- Ba cách giải từ dễ hiểu đến gọn nhất, code chuẩn LeetCode, độ phức tạp.
- Lỗi hay mắc, mẹo nhận dạng bài "bất biến", biến thể và bài tập có đáp án.

# 1. Đề bài nói gì?

Cho hai mảng số nguyên `source` và `target` (cùng độ dài `n`).

Trong **một thao tác**, bạn được:

1. Chọn hai chỉ số **khác nhau** `i` và `j` trong `source`.
2. Chọn một số nguyên **bất kỳ** `delta` (âm, dương, 0 đều được).
3. Cập nhật **đồng thời**:

```text
source[i] = source[i] + source[j] - delta
source[j] = delta
```

Hỏi: có thể biến `source` thành **đúng bằng** `target` sau một số thao tác (kể cả 0 thao tác) hay không? Trả về `True` / `False`.

### Ví dụ

| source | target | Kết quả |
|---|---|---|
| `[1, 2, 3]` | `[0, 2, 4]` | `True` |
| `[-5, -5]` | `[-15, 5]` | `True` |
| `[1, 2, 1]` | `[0, 2, 5]` | `False` |

### Ràng buộc

- `2 <= n <= 10^5`
- `-10^9 <= source[i], target[i] <= 10^9`

> Lưu ý quan trọng: `n >= 2`, nên **luôn** chọn được hai chỉ số khác nhau.

# 2. Viết code mô phỏng đúng một thao tác

Trước khi nghĩ thuật toán, hãy **làm cho phép toán "sống"** bằng code. Đây là thói quen rất tốt với các bài có phép biến đổi lạ: mô phỏng trước, suy nghĩ sau.

Ta viết hàm `apply_operation(arr, i, j, delta)`:

- Nhận vào mảng `arr`, hai chỉ số `i`, `j` và số `delta`.
- Trả về **một mảng mới** sau thao tác (không sửa mảng gốc).

Vì sao trả về mảng mới? Để ta có thể so sánh "trước" và "sau" một cách dễ dàng.

In [1]:
def apply_operation(arr, i, j, delta):
    assert i != j, "i và j phải khác nhau"
    new_arr = arr[:]                     # sao chép mảng
    old_i, old_j = arr[i], arr[j]        # lưu giá trị CŨ trước khi sửa
    new_arr[i] = old_i + old_j - delta
    new_arr[j] = delta
    return new_arr


# Ví dụ 1 của đề: i = 0, j = 2, delta = 4
print(apply_operation([1, 2, 3], 0, 2, 4))      # mong đợi [0, 2, 4]

# Ví dụ 2 của đề: i = 1, j = 0, delta = -15
print(apply_operation([-5, -5], 1, 0, -15))     # mong đợi [-15, 5]

[0, 2, 4]
[-15, 5]


## 2.1. Vì sao phải viết `new_arr = arr[:]`?

Trong Python, câu lệnh `b = a` **không tạo mảng mới**. Nó chỉ cho thêm một "cái tên" `b` trỏ vào **cùng một mảng** với `a`.

Nghĩa là sửa `b` thì `a` cũng bị sửa theo. Hiện tượng này gọi là **aliasing** (hai tên, một đối tượng).

Còn `a[:]` là **slicing** (cắt mảng) lấy từ đầu đến cuối → tạo ra **một bản sao mới**.

Chạy cell dưới để thấy sự khác biệt.

In [2]:
a = [1, 2, 3]
b = a            # b và a là CÙNG một mảng
b[0] = 100
print("a sau khi sửa b:", a)     # a cũng bị đổi!

a = [1, 2, 3]
c = a[:]         # c là bản sao
c[0] = 100
print("a sau khi sửa c:", a)     # a giữ nguyên
print("c:", c)

a sau khi sửa b: [100, 2, 3]
a sau khi sửa c: [1, 2, 3]
c: [100, 2, 3]


## 2.2. Vì sao phải lưu `old_i`, `old_j` trước?

Đề nói cập nhật **đồng thời**. Nếu ta viết:

```python
arr[j] = delta
arr[i] = arr[i] + arr[j] - delta    # SAI: arr[j] lúc này đã là delta rồi!
```

thì dòng thứ hai dùng nhầm giá trị **mới** của `arr[j]`. Kết quả: `arr[i]` không đổi gì cả.

Quy tắc: khi nhiều biến được cập nhật "cùng lúc", hãy **lưu giá trị cũ** trước, rồi mới tính giá trị mới.

In [3]:
def apply_operation_wrong(arr, i, j, delta):
    new_arr = arr[:]
    new_arr[j] = delta
    new_arr[i] = new_arr[i] + new_arr[j] - delta   # dùng nhầm giá trị mới
    return new_arr

print("Sai :", apply_operation_wrong([1, 2, 3], 0, 2, 4))
print("Đúng:", apply_operation([1, 2, 3], 0, 2, 4))

Sai : [1, 2, 4]
Đúng: [0, 2, 4]


# 3. Thí nghiệm: điều gì KHÔNG thay đổi?

Đề cho `delta` là **bất kỳ số nguyên nào** → số cách làm là vô hạn. Ta không thể thử hết.

Khi gặp tình huống "làm phép toán bao nhiêu lần cũng được", câu hỏi vàng là:

> **Có đại lượng nào luôn giữ nguyên sau mỗi thao tác không?**

Đại lượng như vậy gọi là **bất biến (invariant)**.

Cách tìm nhanh nhất cho người mới: **thí nghiệm**. Ta làm ngẫu nhiên nhiều thao tác và in ra vài đại lượng quen thuộc: tổng (`sum`), giá trị lớn nhất (`max`), nhỏ nhất (`min`)...

### Công cụ: module `random`

- `random.randrange(n)` → một số nguyên ngẫu nhiên trong `0, 1, ..., n-1`.
- `random.randint(a, b)` → một số nguyên ngẫu nhiên trong đoạn `[a, b]` (gồm cả `a` và `b`).
- `random.seed(0)` → cố định "hạt giống" để lần chạy nào cũng ra cùng kết quả (dễ đối chiếu).

In [4]:
import random

random.seed(0)
print(random.randrange(5), random.randrange(5), random.randrange(5))
print(random.randint(-10, 10), random.randint(-10, 10))

3 3 0
-2 6


In [5]:
random.seed(42)

arr = [1, 2, 3]
print(f"Ban đầu: {arr}  | sum = {sum(arr)}, max = {max(arr)}, min = {min(arr)}")

for step in range(1, 7):
    i = random.randrange(len(arr))
    j = random.randrange(len(arr))
    while j == i:                       # bắt buộc i khác j
        j = random.randrange(len(arr))
    delta = random.randint(-20, 20)
    arr = apply_operation(arr, i, j, delta)
    print(f"Bước {step}: i={i}, j={j}, delta={delta:>3} -> {arr}  | sum = {sum(arr)}, max = {max(arr)}, min = {min(arr)}")

Ban đầu: [1, 2, 3]  | sum = 6, max = 3, min = 1
Bước 1: i=2, j=0, delta=-19 -> [-19, 2, 23]  | sum = 6, max = 23, min = -19
Bước 2: i=2, j=1, delta= -5 -> [-19, -5, 30]  | sum = 6, max = 30, min = -19
Bước 3: i=0, j=2, delta=-14 -> [25, -5, -14]  | sum = 6, max = 25, min = -14
Bước 4: i=2, j=0, delta= 17 -> [17, -5, -6]  | sum = 6, max = 17, min = -6
Bước 5: i=1, j=0, delta=-19 -> [-19, 31, -6]  | sum = 6, max = 31, min = -19
Bước 6: i=0, j=2, delta= 18 -> [-43, 31, 18]  | sum = 6, max = 31, min = -43


### Quan sát

- `max` và `min` thay đổi lung tung.
- Nhưng **`sum` luôn bằng 6** — đúng bằng tổng ban đầu `1 + 2 + 3`!

Đây là một **dự đoán** (giả thuyết). Thí nghiệm chỉ gợi ý, chưa phải chứng minh. Bước tiếp theo: **chứng minh** nó bằng đại số.

# 4. Chứng minh: tổng là bất biến

Trước thao tác, hai ô bị đụng tới có tổng:

```text
source[i] + source[j]
```

Sau thao tác:

```text
source[i]_mới + source[j]_mới
= (source[i] + source[j] - delta) + delta
= source[i] + source[j]
```

`delta` bị **trừ ở ô i** và **cộng vào ô j** → triệt tiêu. Các ô khác không bị đụng tới.

> Vậy **tổng của cả mảng không bao giờ đổi**, bất kể chọn `i`, `j`, `delta` thế nào.

Một cách hình dung dễ nhớ: thao tác giống như **hai người góp tiền chung**, rồi người `j` lấy về đúng `delta` đồng, người `i` giữ phần còn lại. Tổng tiền của cả nhóm không đổi.

In [6]:
# Kiểm tra lại bằng nhiều thao tác ngẫu nhiên hơn, trên nhiều mảng khác nhau
random.seed(7)

for trial in range(5):
    arr = [random.randint(-50, 50) for _ in range(random.randint(2, 6))]
    original_sum = sum(arr)
    for _ in range(1000):
        i, j = random.sample(range(len(arr)), 2)    # lấy 2 chỉ số khác nhau
        arr = apply_operation(arr, i, j, random.randint(-10**9, 10**9))
        assert sum(arr) == original_sum
    print(f"Thử {trial}: 1000 thao tác, tổng vẫn = {original_sum}  ✔")

Thử 0: 1000 thao tác, tổng vẫn = -42  ✔
Thử 1: 1000 thao tác, tổng vẫn = -24  ✔
Thử 2: 1000 thao tác, tổng vẫn = 8  ✔
Thử 3: 1000 thao tác, tổng vẫn = 98  ✔
Thử 4: 1000 thao tác, tổng vẫn = -26  ✔


Ở cell trên có cú pháp mới:

- `random.sample(range(len(arr)), 2)` → lấy ngẫu nhiên **2 phần tử khác nhau** từ dãy `0..len(arr)-1`. Rất tiện để chọn `i != j`.
- `[... for _ in range(k)]` → **list comprehension**: tạo list gồm `k` phần tử. Dấu `_` là tên biến "không dùng tới".

# 5. Hệ quả 1 — Điều kiện CẦN

Vì tổng không bao giờ đổi:

> Nếu `sum(source) != sum(target)` thì **chắc chắn không thể** biến đổi → trả về `False`.

Kiểm tra ví dụ 3:

In [7]:
source = [1, 2, 1]
target = [0, 2, 5]
print("sum(source) =", sum(source))
print("sum(target) =", sum(target))
print("Có thể biến đổi?", sum(source) == sum(target))

sum(source) = 4
sum(target) = 7
Có thể biến đổi? False


# 6. Câu hỏi ngược: tổng bằng nhau thì CHẮC CHẮN làm được không?

Đây là bước nhiều người bỏ qua. "Tổng bằng nhau" mới chỉ là điều kiện **cần**. Ta phải chỉ ra nó cũng là điều kiện **đủ**, tức là **luôn tồn tại** cách biến đổi.

## Nhìn lại thao tác theo cách khác

```text
source[j] = delta                     # ô j nhận BẤT KỲ giá trị nào ta muốn
source[i] = source[i] + source[j] - delta   # ô i "gánh" phần chênh lệch
```

Nói cách khác:

> Một thao tác = **đặt ô `j` thành giá trị tuỳ ý**, còn phần chênh lệch được **dồn vào ô `i`**.

## Ý tưởng "neo" (anchor)

- Chọn ô `0` làm **neo** — ô chuyên đi "gánh" chênh lệch.
- Với mỗi ô `j = 1, 2, ..., n-1`: làm thao tác `(i = 0, j, delta = target[j])` → ô `j` thành **đúng** `target[j]`.
- Sau khi xử lý xong mọi ô `j >= 1`, chỉ còn ô `0` là chưa "được chỉnh".

Ô `0` lúc đó bằng bao nhiêu? Vì tổng không đổi:

```text
source[0]_cuối = tổng_ban_đầu - (target[1] + ... + target[n-1])
```

Nếu `sum(source) == sum(target)` thì:

```text
source[0]_cuối = sum(target) - (target[1] + ... + target[n-1]) = target[0]
```

→ Ô neo **tự động** đúng! Không cần làm gì thêm.

In [8]:
def transform_with_anchor(source, target):
    """Thực hiện chiến lược 'neo ở ô 0'. Trả về (mảng cuối, danh sách thao tác)."""
    arr = source[:]
    operations = []
    for j in range(1, len(arr)):
        if arr[j] != target[j]:              # ô đã đúng thì bỏ qua
            delta = target[j]
            arr = apply_operation(arr, 0, j, delta)
            operations.append((0, j, delta))
    return arr, operations


final, ops = transform_with_anchor([1, 2, 3], [0, 2, 4])
print("Mảng cuối:", final)
print("Các thao tác (i, j, delta):", ops)

Mảng cuối: [0, 2, 4]
Các thao tác (i, j, delta): [(0, 2, 4)]


# 7. Mô phỏng từng bước (trace)

Lấy ví dụ dài hơn để thấy rõ ô neo "gánh" chênh lệch như thế nào:

```text
source = [5, -1, 4, 2]     tổng = 10
target = [3,  6, 0, 1]     tổng = 10
```

| Bước | Thao tác `(i, j, delta)` | Ô 0 (neo) trước → sau | Ô j trước → sau | Mảng sau thao tác |
|---|---|---|---|---|
| 1 | `(0, 1, 6)` | `5 → 5 + (-1) - 6 = -2` | `-1 → 6` | `[-2, 6, 4, 2]` |
| 2 | `(0, 2, 0)` | `-2 → -2 + 4 - 0 = 2` | `4 → 0` | `[2, 6, 0, 2]` |
| 3 | `(0, 3, 1)` | `2 → 2 + 2 - 1 = 3` | `2 → 1` | `[3, 6, 0, 1]` ✔ |

Sau bước 3, ô neo bằng `3 = target[0]` — đúng như chứng minh.

Hàm `trace_anchor` dưới đây in ra đúng bảng này.

In [9]:
def trace_anchor(source, target):
    arr = source[:]
    print(f"source = {source} (tổng {sum(source)})")
    print(f"target = {target} (tổng {sum(target)})")
    print("-" * 60)
    for j in range(1, len(arr)):
        if arr[j] == target[j]:
            print(f"j={j}: ô {j} đã đúng ({arr[j]}), bỏ qua")
            continue
        before_anchor, before_j = arr[0], arr[j]
        arr = apply_operation(arr, 0, j, target[j])
        print(f"j={j}: thao tác (0, {j}, {target[j]:>3}) | neo {before_anchor:>3} -> {arr[0]:>3} | "
              f"ô {j}: {before_j:>3} -> {arr[j]:>3} | mảng = {arr}")
    print("-" * 60)
    print("Kết quả cuối:", arr, "| bằng target?", arr == target)
    return arr == target


trace_anchor([5, -1, 4, 2], [3, 6, 0, 1])

source = [5, -1, 4, 2] (tổng 10)
target = [3, 6, 0, 1] (tổng 10)
------------------------------------------------------------
j=1: thao tác (0, 1,   6) | neo   5 ->  -2 | ô 1:  -1 ->   6 | mảng = [-2, 6, 4, 2]
j=2: thao tác (0, 2,   0) | neo  -2 ->   2 | ô 2:   4 ->   0 | mảng = [2, 6, 0, 2]
j=3: thao tác (0, 3,   1) | neo   2 ->   3 | ô 3:   2 ->   1 | mảng = [3, 6, 0, 1]
------------------------------------------------------------
Kết quả cuối: [3, 6, 0, 1] | bằng target? True


True

In [10]:
# Khi tổng KHÁC nhau, chiến lược neo vẫn chạy được, nhưng ô neo cuối cùng bị sai
trace_anchor([1, 2, 1], [0, 2, 5])

source = [1, 2, 1] (tổng 4)
target = [0, 2, 5] (tổng 7)
------------------------------------------------------------
j=1: ô 1 đã đúng (2), bỏ qua
j=2: thao tác (0, 2,   5) | neo   1 ->  -3 | ô 2:   1 ->   5 | mảng = [-3, 2, 5]
------------------------------------------------------------
Kết quả cuối: [-3, 2, 5] | bằng target? False


False

# 8. Cách 1 — Kiểm chứng "thô": BFS vét cạn trên miền nhỏ

Lập luận ở trên đã đủ chặt. Nhưng người mới thường muốn **thấy tận mắt**: "nếu thử hết mọi khả năng thì có đúng là chỉ cần so sánh tổng không?"

Không thể thử hết vì `delta` vô hạn. Nhưng ta có thể thử **trên miền nhỏ**: mảng dài 3, mọi giá trị giới hạn trong `[-B, B]`.

## BFS (Breadth-First Search — tìm kiếm theo chiều rộng) là gì?

Hãy tưởng tượng mỗi mảng là một **trạng thái**. Từ một trạng thái, làm một thao tác sẽ sang trạng thái khác.

BFS đi "loang" dần:

1. Bỏ trạng thái ban đầu vào **hàng đợi** (`queue`).
2. Lấy một trạng thái ra, thử mọi thao tác, trạng thái mới nào **chưa thăm** thì đánh dấu và bỏ vào hàng đợi.
3. Lặp đến khi hàng đợi rỗng → tập `visited` chính là **mọi trạng thái đi tới được**.

Hai công cụ Python:

- `collections.deque` — hàng đợi, `append` thêm vào cuối, `popleft` lấy ra từ đầu, đều nhanh.
- `set` — tập hợp, kiểm tra `x in visited` rất nhanh. Vì `list` không bỏ vào `set` được, ta đổi mảng thành `tuple`.

In [11]:
from collections import deque

q = deque()
q.append((1, 2, 3))
q.append((0, 2, 4))
print("Lấy ra:", q.popleft())     # (1, 2, 3) — vào trước ra trước
print("Còn lại:", q)

visited = {(1, 2, 3)}
print((1, 2, 3) in visited, (9, 9, 9) in visited)

Lấy ra: (1, 2, 3)
Còn lại: deque([(0, 2, 4)])
True False


In [12]:
def reachable_states_bfs(source, bound):
    """Mọi trạng thái đi tới được, với điều kiện mọi phần tử luôn nằm trong [-bound, bound]."""
    n = len(source)
    start = tuple(source)
    visited = {start}
    queue = deque([start])
    while queue:
        state = queue.popleft()
        for i in range(n):
            for j in range(n):
                if i == j:
                    continue
                for delta in range(-bound, bound + 1):
                    new_state = list(state)
                    new_state[i] = state[i] + state[j] - delta
                    new_state[j] = delta
                    if abs(new_state[i]) > bound:
                        continue            # ra ngoài miền nhỏ thì bỏ
                    new_state = tuple(new_state)
                    if new_state not in visited:
                        visited.add(new_state)
                        queue.append(new_state)
    return visited


reach = reachable_states_bfs([1, 2, 1], bound=4)
print("Số trạng thái đi tới được:", len(reach))
print("Tổng của các trạng thái đi tới được:", {sum(s) for s in reach})
print("[0, 2, 5] đi tới được?", (0, 2, 5) in reach)
print("[0, 0, 4] đi tới được?", (0, 0, 4) in reach)

Số trạng thái đi tới được: 45
Tổng của các trạng thái đi tới được: {4}
[0, 2, 5] đi tới được? False
[0, 0, 4] đi tới được? True


### Đối chiếu BFS với quy tắc "so sánh tổng"

Ta lấy **mọi** `target` có các phần tử trong `[-2, 2]`, và so:

- BFS nói đi tới được hay không (cho phép giá trị trung gian rộng hơn: `[-6, 6]`).
- Quy tắc nói `sum(source) == sum(target)`.

Nếu hai cách luôn khớp nhau → bằng chứng thực nghiệm rất mạnh cho lời giải.

In [13]:
from itertools import product

def check_rule_against_bfs(source):
    reach = reachable_states_bfs(source, bound=6)
    mismatch = 0
    for target in product(range(-2, 3), repeat=len(source)):   # mọi target trong [-2,2]^n
        by_bfs = target in reach
        by_rule = sum(source) == sum(target)
        if by_bfs != by_rule:
            mismatch += 1
    return mismatch

for src in [[1, 2, 1], [0, 0, 0], [-2, 2, 1], [2, -1]]:
    print(f"source = {src}: số target mà BFS và quy tắc 'tổng' khác nhau = {check_rule_against_bfs(src)}")

source = [1, 2, 1]: số target mà BFS và quy tắc 'tổng' khác nhau = 0
source = [0, 0, 0]: số target mà BFS và quy tắc 'tổng' khác nhau = 0
source = [-2, 2, 1]: số target mà BFS và quy tắc 'tổng' khác nhau = 0
source = [2, -1]: số target mà BFS và quy tắc 'tổng' khác nhau = 0


`itertools.product(range(-2, 3), repeat=3)` sinh ra **mọi bộ 3 số**, mỗi số trong `-2..2` (tổng cộng `5^3 = 125` bộ). Đây là cách liệt kê vét cạn rất gọn.

Kết quả **0 sai khác** ở mọi trường hợp → quy tắc đúng (trên miền đã thử).

### Nhược điểm của cách BFS

- Chỉ chạy được với mảng rất ngắn và giá trị rất nhỏ (số trạng thái tăng theo hàm mũ).
- Hoàn toàn không dùng được cho `n = 10^5`, giá trị `10^9`.

→ BFS là **công cụ kiểm chứng**, không phải lời giải nộp bài.

# 9. Cách 2 — Mô phỏng chiến lược "neo" (O(n))

Đây là cách "xây dựng" (constructive): thực sự làm các thao tác rồi kiểm tra mảng cuối có bằng `target` không.

Ưu điểm: rất **trực quan**, đúng với lập luận ở mục 6. Không cần tin vào công thức, ta thấy tận mắt.

Để chạy nhanh, ta không tạo mảng mới mỗi lần, mà chỉ giữ **giá trị của ô neo** trong một biến `anchor`:

- Thao tác `(0, j, target[j])` làm `anchor = anchor + arr[j] - target[j]`, và ô `j` thành `target[j]`.
- Cuối cùng chỉ cần kiểm tra `anchor == target[0]`.

In [14]:
def can_transform_simulation(source, target):
    anchor = source[0]
    for j in range(1, len(source)):
        # ô j được đặt thành target[j]; phần chênh lệch dồn vào neo
        anchor = anchor + source[j] - target[j]
    return anchor == target[0]


print(can_transform_simulation([1, 2, 3], [0, 2, 4]))     # True
print(can_transform_simulation([-5, -5], [-15, 5]))       # True
print(can_transform_simulation([1, 2, 1], [0, 2, 5]))     # False

True
True
False


Nhìn kỹ công thức của `anchor` sau vòng lặp:

```text
anchor = source[0] + (source[1] - target[1]) + ... + (source[n-1] - target[n-1])
```

Điều kiện `anchor == target[0]` tương đương:

```text
source[0] + source[1] + ... + source[n-1] == target[0] + target[1] + ... + target[n-1]
```

Tức là... **so sánh tổng**! Cách 2 tự dẫn ta tới cách 3.

# 10. Cách 3 — So sánh tổng (gọn nhất)

```python
return sum(source) == sum(target)
```

Một dòng, nhưng đằng sau là hai chiều lập luận:

| Chiều | Nội dung | Dựa vào |
|---|---|---|
| Cần (⇒) | Biến đổi được ⇒ tổng bằng nhau | Tổng là bất biến (mục 4) |
| Đủ (⇐) | Tổng bằng nhau ⇒ biến đổi được | Chiến lược neo (mục 6) |

In [15]:
def can_transform_sum(source, target):
    return sum(source) == sum(target)


# Đối chiếu cách 2 và cách 3 trên dữ liệu ngẫu nhiên
random.seed(1)
for _ in range(2000):
    n = random.randint(2, 6)
    s = [random.randint(-5, 5) for _ in range(n)]
    t = [random.randint(-5, 5) for _ in range(n)]
    assert can_transform_simulation(s, t) == can_transform_sum(s, t)
print("Cách 2 và cách 3 luôn cho cùng kết quả ✔")

Cách 2 và cách 3 luôn cho cùng kết quả ✔


# 11. Lời giải LeetCode chuẩn

Ta đưa logic vào đúng class và chữ ký hàm mà đề yêu cầu.

In [16]:
class Solution:
    def canTransform(self, source: list[int], target: list[int]) -> bool:
        # Thao tác giữ nguyên tổng (bất biến), và khi n >= 2,
        # tổng bằng nhau là đủ để dựng được target bằng chiến lược "neo".
        return sum(source) == sum(target)

In [17]:
sol = Solution()

tests = [
    ([1, 2, 3], [0, 2, 4], True),          # ví dụ 1
    ([-5, -5], [-15, 5], True),            # ví dụ 2
    ([1, 2, 1], [0, 2, 5], False),         # ví dụ 3
    ([7, 7], [7, 7], True),                # đã bằng nhau, 0 thao tác
    ([3, 1], [1, 3], True),                # hoán đổi
    ([10**9] * 5, [-10**9, 10**9, 10**9, 10**9, 3 * 10**9], True),   # giá trị lớn
    ([0, 0, 0], [1, 0, 0], False),         # lệch tổng 1 đơn vị
]

for source, target, expected in tests:
    result = sol.canTransform(source, target)
    print(f"{str(source):38} {str(target):45} -> {result!s:5} | expected {expected!s:5} | {'PASS' if result == expected else 'FAIL'}")
    assert result == expected

[1, 2, 3]                              [0, 2, 4]                                     -> True  | expected True  | PASS
[-5, -5]                               [-15, 5]                                      -> True  | expected True  | PASS
[1, 2, 1]                              [0, 2, 5]                                     -> False | expected False | PASS
[7, 7]                                 [7, 7]                                        -> True  | expected True  | PASS
[3, 1]                                 [1, 3]                                        -> True  | expected True  | PASS
[1000000000, 1000000000, 1000000000, 1000000000, 1000000000] [-1000000000, 1000000000, 1000000000, 1000000000, 3000000000] -> True  | expected True  | PASS
[0, 0, 0]                              [1, 0, 0]                                     -> False | expected False | PASS


# 12. Vì sao thuật toán đúng? (tóm tắt chứng minh)

**Bất biến:** Gọi `S = tổng các phần tử`. Mỗi thao tác chỉ đổi hai ô `i`, `j`, và tổng của hai ô đó giữ nguyên `(a + b - delta) + delta = a + b`. Vậy `S` không đổi sau mọi thao tác.

**Chiều cần:** Nếu biến được `source` thành `target` thì `sum(source) = S = sum(target)`.

**Chiều đủ:** Nếu `sum(source) = sum(target)`, làm `n - 1` thao tác `(0, j, target[j])` với `j = 1..n-1`. Mỗi ô `j >= 1` thành `target[j]` và không bị đụng lại. Ô 0 cuối cùng bằng `S - (target[1] + ... + target[n-1]) = target[0]`.

**Vai trò của `n >= 2`:** Cần ít nhất 2 ô để có cặp `i != j`. Nếu `n = 1` thì không có thao tác nào, đáp án sẽ là `source == target` (nhưng đề đã loại trường hợp này).

# 13. Độ phức tạp

Gọi `n = len(source)`.

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 1. BFS vét cạn | hàm mũ theo `n` và miền giá trị | rất lớn | chỉ để kiểm chứng |
| 2. Mô phỏng neo | `O(n)` | `O(1)` | trực quan |
| 3. So sánh tổng | `O(n)` | `O(1)` | gọn nhất |

`sum()` phải đọc qua từng phần tử nên tốn `O(n)`; không thể nhanh hơn vì ta buộc phải đọc dữ liệu.

> Ghi chú cho C++/Java: tổng có thể tới `10^5 × 10^9 = 10^14` → phải dùng `long long` / `long`. Python tự xử lý số lớn nên không lo tràn số.

# 14. Những lỗi thường gặp

## Lỗi 1 — So sánh "multiset" (sắp xếp rồi so)

```python
return sorted(source) == sorted(target)     # SAI
```

Ví dụ 1: `sorted([1,2,3]) = [1,2,3]` khác `sorted([0,2,4]) = [0,2,4]`, nhưng đáp án là `True`. Thao tác này **tạo ra giá trị mới**, không chỉ hoán đổi.

## Lỗi 2 — Chỉ chứng minh một chiều

"Tổng không đổi, vậy so tổng là xong" — mới đúng **một nửa**. Nếu không chỉ ra cách dựng (chiều đủ), bạn không chắc tổng bằng nhau là đủ. Ở nhiều bài khác, còn có **thêm** bất biến (ví dụ tính chẵn lẻ) và chỉ so tổng sẽ sai.

## Lỗi 3 — Cập nhật tuần tự khi mô phỏng

Như mục 2.2: gán `source[j] = delta` trước rồi mới tính `source[i]` → sai.

## Lỗi 4 — Nghĩ rằng `delta` phải "nằm trong mảng" hay "không âm"

Đề cho `delta` là **bất kỳ số nguyên nào**. Chính sự tự do này làm điều kiện đủ trở nên dễ.

In [18]:
# Minh hoạ lỗi 1
source, target = [1, 2, 3], [0, 2, 4]
print("So sorted (SAI):", sorted(source) == sorted(target))
print("So tổng (ĐÚNG) :", sum(source) == sum(target))

So sorted (SAI): False
So tổng (ĐÚNG) : True


# 15. Mẹo nhận diện bài "tìm bất biến"

Dấu hiệu:

- Đề cho một **phép biến đổi lạ** và hỏi "có thể biến A thành B không?" / "làm bao nhiêu lần cũng được".
- Tham số tự do (như `delta` bất kỳ) → không thể thử hết.

Quy trình 4 bước:

1. **Mô phỏng** phép biến đổi bằng code.
2. **Thí nghiệm** ngẫu nhiên, in các đại lượng: tổng, tích, XOR, chẵn/lẻ, min/max, multiset, GCD...
3. **Chứng minh** đại lượng nào không đổi → điều kiện **cần**.
4. **Dựng cách biến đổi** khi điều kiện thoả → điều kiện **đủ**. Kỹ thuật "neo" (dồn mọi chênh lệch vào một ô) rất hay dùng.

| Phép biến đổi | Bất biến thường gặp |
|---|---|
| `a[i] += 1, a[j] -= 1` | tổng |
| `a[i] ^= x, a[j] ^= x` | XOR toàn mảng |
| hoán đổi hai phần tử | multiset (tần suất từng giá trị) |
| `a[i] += 2` hoặc `-= 2` | tính chẵn lẻ của từng phần tử |

# 16. Biến thể 1 — Phiên bản XOR

Giả sử thao tác đổi thành:

```text
source[i] = source[i] XOR source[j] XOR delta
source[j] = delta
```

(`XOR`, ký hiệu `^` trong Python, là phép "cộng không nhớ" trên từng bit.)

Hai tính chất cần biết: `x ^ x = 0` và `x ^ 0 = x`.

Khi đó `source[i]_mới ^ source[j]_mới = source[i] ^ source[j] ^ delta ^ delta = source[i] ^ source[j]`.

→ **XOR của cả mảng** là bất biến. Và chiến lược neo vẫn dùng được y hệt. Vậy đáp án là: `XOR(source) == XOR(target)`.

In [19]:
from functools import reduce
from operator import xor

print(5 ^ 5, 5 ^ 0, 6 ^ 3)      # 0, 5, 5

def xor_all(arr):
    return reduce(xor, arr, 0)   # 0 ^ arr[0] ^ arr[1] ^ ...

def can_transform_xor_version(source, target):
    return xor_all(source) == xor_all(target)

print(can_transform_xor_version([1, 2, 3], [0, 0, 0]))   # 1^2^3 = 0 -> True
print(can_transform_xor_version([1, 2], [3, 1]))         # 3 vs 2 -> False

0 5 5
True
False


`reduce(f, arr, init)` "gộp" cả mảng bằng hàm `f`: `f(f(f(init, arr[0]), arr[1]), arr[2])...`. Với `f = xor` và `init = 0`, ta được XOR của mọi phần tử.

# 17. Biến thể 2 — Chỉ được hoán đổi

Nếu thao tác chỉ là "hoán đổi `source[i]` và `source[j]`", thì không có giá trị mới nào được tạo ra. Bất biến lúc này là **multiset** (mỗi giá trị xuất hiện bao nhiêu lần). Đáp án: `Counter(source) == Counter(target)` (hoặc so `sorted`).

Đây chính là chỗ "Lỗi 1" ở trên đến từ: nhầm bài này với biến thể hoán đổi.

In [20]:
from collections import Counter

def can_transform_swap_only(source, target):
    return Counter(source) == Counter(target)

print(can_transform_swap_only([3, 1, 2], [1, 2, 3]))   # True
print(can_transform_swap_only([1, 2, 3], [0, 2, 4]))   # False (dù tổng bằng nhau)

True
False


# 18. Template tư duy

```python
def can_reach(start, goal):
    # 1. Tìm bất biến I(.) của phép biến đổi
    # 2. Điều kiện cần: I(start) == I(goal)
    # 3. Dựng cách biến đổi khi điều kiện thoả (thường: chọn một phần tử làm "neo")
    return I(start) == I(goal)
```

Với bài này: `I(arr) = sum(arr)`.

# 19. Bài tập vận dụng

## Bài 1 — Tính bằng tay (không chạy code)

Với mỗi cặp, trả lời `True`/`False`:

1. `source = [4, 4]`, `target = [8, 0]`
2. `source = [1, -1, 0]`, `target = [0, 0, 1]`
3. `source = [2, 3, 5]`, `target = [10, 0, 0]`
4. `source = [-3, 3, -3, 3]`, `target = [0, 0, 0, 1]`

## Bài 2 — Dự đoán output

Không chạy code, hãy đoán:

```python
apply_operation([1, 2, 3], 2, 0, 10)
apply_operation([0, 0], 0, 1, -7)
```

Sau đó chạy cell kiểm tra ngay bên dưới.

In [21]:
print(apply_operation([1, 2, 3], 2, 0, 10))
print(apply_operation([0, 0], 0, 1, -7))

[10, 2, -6]
[7, -7]


## Bài 3 — Điền chỗ trống

Hoàn thiện hàm mô phỏng chiến lược neo, nhưng lần này **chọn ô cuối cùng (`n - 1`) làm neo**.

In [22]:
def exercise_3_anchor_last(source, target):
    n = len(source)
    # TODO 1: anchor = giá trị ban đầu của ô cuối
    # TODO 2: duyệt j = 0 .. n-2, dồn chênh lệch (source[j] - target[j]) vào anchor
    # TODO 3: trả về anchor == target[n - 1]
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")

# Test sau khi làm xong:
# assert exercise_3_anchor_last([1, 2, 3], [0, 2, 4]) is True
# assert exercise_3_anchor_last([1, 2, 1], [0, 2, 5]) is False

## Bài 4 — Trả về danh sách thao tác

Viết hàm trả về **danh sách các thao tác** `(i, j, delta)` biến `source` thành `target`, hoặc `None` nếu không thể. Sau đó tự kiểm tra bằng cách áp dụng lần lượt các thao tác với `apply_operation`.

In [23]:
def exercise_4_build_operations(source, target):
    # TODO:
    # - nếu tổng khác nhau -> return None
    # - ngược lại: với mỗi j >= 1 mà source[j] != target[j], thêm (0, j, target[j])
    raise NotImplementedError

# ops = exercise_4_build_operations([5, -1, 4, 2], [3, 6, 0, 1])
# arr = [5, -1, 4, 2]
# for i, j, d in ops:
#     arr = apply_operation(arr, i, j, d)
# assert arr == [3, 6, 0, 1]

## Bài 5 — Kiểm chứng biến thể XOR bằng BFS

Ở mục 16 ta khẳng định: với thao tác XOR, biến được khi và chỉ khi `XOR(source) == XOR(target)`.

Hãy viết hàm `exercise_5_reachable_xor(source)` dùng BFS để tìm **mọi** trạng thái đi tới được, với mọi giá trị và `delta` nằm trong `0..7`.

Câu hỏi phụ: vì sao với XOR, ta **không cần** cắt bỏ trạng thái "ra ngoài miền" như ở mục 8?

In [24]:
def exercise_5_reachable_xor(source):
    # TODO:
    # - visited = {tuple(source)}, queue = deque([tuple(source)])
    # - với mỗi trạng thái lấy ra: thử mọi i != j, mọi delta trong range(8)
    #   new[i] = state[i] ^ state[j] ^ delta ; new[j] = delta
    # - trạng thái mới chưa thăm thì thêm vào visited và queue
    raise NotImplementedError

## Bài 6 — Mini challenge: ít thao tác

Viết hàm trả về danh sách thao tác có **không quá `max(0, m - 1)` thao tác**, với `m` là số vị trí mà `source[k] != target[k]`.

Gợi ý: đừng cố định neo ở ô 0. Nếu ô 0 vốn đã đúng mà vẫn làm neo, nó sẽ bị "làm hỏng" rồi phải chỉnh lại...

In [25]:
def exercise_6_few_operations(source, target):
    # TODO
    raise NotImplementedError

## Gợi ý theo mức độ

**Bài 3**
- Mức 1: giống hệt cách 2, chỉ đổi vai trò ô 0 ↔ ô `n-1`.
- Mức 2: `for j in range(n - 1):`
- Mức 3: `anchor = anchor + source[j] - target[j]`

**Bài 5**
- Mức 1: chép lại hàm `reachable_states_bfs` ở mục 8, chỉ đổi công thức cập nhật.
- Mức 2: `new_state[i] = state[i] ^ state[j] ^ delta`.
- Mức 3: XOR của hai số trong `0..7` (3 bit) vẫn nằm trong `0..7` → miền tự "đóng", không cần kiểm tra biên.

**Bài 6**
- Mức 1: chọn neo là **một vị trí đang sai**.
- Mức 2: chỉ làm thao tác cho các vị trí sai khác neo → `m - 1` thao tác.
- Mức 3: nếu tổng bằng nhau thì không thể có đúng 1 vị trí sai (vì sao?).

# 20. Đáp án tham khảo

Chỉ xem sau khi đã tự làm.

**Bài 1:**

| Câu | Tổng source | Tổng target | Đáp án |
|---|---:|---:|---|
| 1 | 8 | 8 | `True` |
| 2 | 0 | 1 | `False` |
| 3 | 10 | 10 | `True` |
| 4 | 0 | 1 | `False` |

**Bài 2:**

- `apply_operation([1, 2, 3], 2, 0, 10)`: ở đây `i = 2`, `j = 0`. Ô 2 = `3 + 1 - 10 = -6`, ô 0 = `10` → `[10, 2, -6]`.
- `apply_operation([0, 0], 0, 1, -7)`: ô 0 = `0 + 0 - (-7) = 7`, ô 1 = `-7` → `[7, -7]`.

> Bẫy hay gặp ở Bài 2: nhầm vai trò `i` và `j`. Ô `j` mới là ô nhận `delta`.

In [26]:
def answer_3_anchor_last(source, target):
    n = len(source)
    anchor = source[n - 1]
    for j in range(n - 1):
        anchor = anchor + source[j] - target[j]
    return anchor == target[n - 1]

assert answer_3_anchor_last([1, 2, 3], [0, 2, 4]) is True
assert answer_3_anchor_last([-5, -5], [-15, 5]) is True
assert answer_3_anchor_last([1, 2, 1], [0, 2, 5]) is False
print("Đáp án bài 3 PASS")

Đáp án bài 3 PASS


In [27]:
def answer_4_build_operations(source, target):
    if sum(source) != sum(target):
        return None
    ops = []
    for j in range(1, len(source)):
        if source[j] != target[j]:
            ops.append((0, j, target[j]))
    return ops

ops = answer_4_build_operations([5, -1, 4, 2], [3, 6, 0, 1])
arr = [5, -1, 4, 2]
for i, j, d in ops:
    arr = apply_operation(arr, i, j, d)
print("Thao tác:", ops, "-> mảng cuối:", arr)
assert arr == [3, 6, 0, 1]
assert answer_4_build_operations([1, 2, 1], [0, 2, 5]) is None
print("Đáp án bài 4 PASS")

Thao tác: [(0, 1, 6), (0, 2, 0), (0, 3, 1)] -> mảng cuối: [3, 6, 0, 1]
Đáp án bài 4 PASS


### Đáp án bài 5

Câu hỏi phụ: mọi giá trị đều có 3 bit, XOR của các số 3 bit vẫn là số 3 bit → không bao giờ "ra ngoài" `0..7`. Vì vậy BFS trên miền này là **chính xác tuyệt đối**, không phải xấp xỉ.

In [28]:
def answer_5_reachable_xor(source):
    n = len(source)
    start = tuple(source)
    visited = {start}
    queue = deque([start])
    while queue:
        state = queue.popleft()
        for i in range(n):
            for j in range(n):
                if i == j:
                    continue
                for delta in range(8):
                    new_state = list(state)
                    new_state[i] = state[i] ^ state[j] ^ delta
                    new_state[j] = delta
                    new_state = tuple(new_state)
                    if new_state not in visited:
                        visited.add(new_state)
                        queue.append(new_state)
    return visited


bad = 0
for src in [[1, 2, 3], [0, 0, 0], [7, 5, 1], [6, 3]]:
    reach = answer_5_reachable_xor(src)
    for tgt in product(range(8), repeat=len(src)):
        if (tgt in reach) != can_transform_xor_version(src, list(tgt)):
            bad += 1
print("Số trường hợp BFS khác quy tắc XOR:", bad)

Số trường hợp BFS khác quy tắc XOR: 0


In [29]:
def answer_6_few_operations(source, target):
    if sum(source) != sum(target):
        return None
    wrong = [k for k in range(len(source)) if source[k] != target[k]]
    if not wrong:
        return []
    anchor = wrong[0]                      # neo là một vị trí đang sai
    return [(anchor, j, target[j]) for j in wrong[1:]]

src, tgt = [7, 1, 5, 3], [7, 4, 2, 3]       # chỉ 2 vị trí sai -> 1 thao tác
ops = answer_6_few_operations(src, tgt)
arr = src[:]
for i, j, d in ops:
    arr = apply_operation(arr, i, j, d)
print("Thao tác:", ops, "-> mảng cuối:", arr)
assert arr == tgt and len(ops) == 1
print("Đáp án bài 6 PASS")

Thao tác: [(1, 2, 2)] -> mảng cuối: [7, 4, 2, 3]
Đáp án bài 6 PASS


# 21. Tổng kết

## Kiến thức Python đã dùng

```python
arr[:]                      # sao chép list
sum(arr)                    # tổng
random.randint / random.sample / random.seed
collections.deque, set      # BFS
itertools.product           # liệt kê vét cạn
functools.reduce, operator.xor
collections.Counter
```

## Kỹ thuật thuật toán

1. **Mô phỏng** phép biến đổi lạ để hiểu nó.
2. **Thí nghiệm** → đoán bất biến.
3. **Chứng minh** bất biến → điều kiện cần.
4. **Dựng cách biến đổi** (chiến lược neo) → điều kiện đủ.

## Công thức nhớ nhanh

> **Thao tác giữ nguyên tổng; có ô neo gánh chênh lệch ⇒ chỉ cần `sum(source) == sum(target)`.**

## Độ phức tạp

```text
Time:  O(n)
Space: O(1)
```

## Checklist trước khi nộp

- [ ] Đã chứng minh **cả hai chiều** (cần và đủ)?
- [ ] Không nhầm với biến thể "hoán đổi" (so sorted)?
- [ ] Tên hàm đúng `canTransform`, trả về `bool`?
- [ ] (C++/Java) dùng kiểu 64-bit cho tổng?

# 22. Phiên bản cuối cùng nên nhớ

```python
class Solution:
    def canTransform(self, source: list[int], target: list[int]) -> bool:
        return sum(source) == sum(target)
```

Một dòng code — nhưng thứ bạn thực sự học được là **quy trình tìm bất biến**: mô phỏng → thí nghiệm → chứng minh cần → dựng cách làm cho đủ.